In [ ]:
import librosa
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import noisereduce as nr
from scipy import signal
from scipy.signal import butter, filtfilt

# 프로젝트 루트 설정
project_root = Path('..').resolve()
print(f"프로젝트 루트: {project_root}")

# 오디오 파일 경로
audio_file = project_root / "mild_video_clip0001.wav"


In [ ]:
def quick_noise_check(audio_file):
    """
    빠른 노이즈 레벨 체크
    """
    y, sr = librosa.load(audio_file, sr=22050, duration=60)  # 1분만 로드

    # 기본 통계
    rms = np.sqrt(np.mean(y**2))
    max_amp = np.max(np.abs(y))

    print("=== 오디오 품질 체크 ===")
    print(f"RMS 레벨: {rms:.4f}")
    print(f"최대 진폭: {max_amp:.4f}")
    print(f"다이나믹 레인지: {20*np.log10(max_amp/rms):.1f} dB")

    # 노이즈 심각도 판단
    if rms > 0.1:
        print("🔴 노이즈 매우 심함 - emergency_noise_reduction 사용")
    elif rms > 0.05:
        print("🟡 노이즈 심함 - aggressive_noise_reduction 사용")
    else:
        print("🟢 노이즈 보통 - 일반 처리 가능")

    return rms, max_amp

In [ ]:
def segment_noise_analysis(audio_file, start_minutes=0, end_minutes=None, duration_minutes=None, sr=22050):
    """
    지정된 구간의 노이즈 레벨을 상세 분석
    
    Parameters:
    - audio_file: 오디오 파일 경로
    - start_minutes: 시작 시간 (분)
    - end_minutes: 종료 시간 (분) - duration_minutes와 둘 중 하나만 사용
    - duration_minutes: 지속 시간 (분) - end_minutes와 둘 중 하나만 사용
    - sr: 샘플링 레이트
    
    Example:
    # 10분부터 30분까지 분석
    segment_noise_analysis('audio.wav', start_minutes=10, end_minutes=30)
    
    # 10분부터 5분 동안 분석  
    segment_noise_analysis('audio.wav', start_minutes=10, duration_minutes=5)
    """
    
    print(f"🔍 구간별 노이즈 분석 시작")
    print(f"파일: {audio_file}")
    
    # 오디오 로드 (전체 파일 메타데이터 먼저 확인)
    y_full, sr = librosa.load(audio_file, sr=sr)
    total_duration_minutes = len(y_full) / sr / 60
    
    print(f"전체 파일 길이: {total_duration_minutes:.1f}분 ({len(y_full)/sr/3600:.1f}시간)")
    
    # 구간 계산
    start_sample = int(start_minutes * 60 * sr)
    
    if end_minutes is not None:
        end_sample = int(end_minutes * 60 * sr)
        actual_duration = end_minutes - start_minutes
    elif duration_minutes is not None:
        end_sample = start_sample + int(duration_minutes * 60 * sr)
        actual_duration = duration_minutes
    else:
        # 끝까지 분석
        end_sample = len(y_full)
        actual_duration = (len(y_full) - start_sample) / sr / 60
    
    # 범위 체크
    end_sample = min(end_sample, len(y_full))
    
    print(f"분석 구간: {start_minutes:.1f}분 ~ {start_minutes + actual_duration:.1f}분 (총 {actual_duration:.1f}분)")
    
    # 해당 구간만 추출
    y_segment = y_full[start_sample:end_sample]
    segment_duration = len(y_segment) / sr
    
    if len(y_segment) == 0:
        print("❌ 오류: 지정된 구간이 비어있거나 범위를 벗어났습니다.")
        return None
    
    print(f"추출된 구간 길이: {segment_duration/60:.1f}분")
    
    return analyze_noise_characteristics(y_segment, sr, start_minutes)

def analyze_noise_characteristics(y_segment, sr, start_time_minutes=0):
    """
    추출된 오디오 구간의 노이즈 특성을 상세 분석
    """
    
    print("\\n📊 === 노이즈 특성 분석 결과 ===")
    
    # 1. 기본 통계
    rms_level = np.sqrt(np.mean(y_segment**2))
    max_amplitude = np.max(np.abs(y_segment))
    dynamic_range = 20 * np.log10(max_amplitude / (rms_level + 1e-10))
    
    print(f"🔊 음량 분석:")
    print(f"   - RMS 레벨: {rms_level:.4f}")
    print(f"   - 최대 진폭: {max_amplitude:.4f}")
    print(f"   - 다이내믹 레인지: {dynamic_range:.1f} dB")
    
    # 2. 주파수 분석
    stft = librosa.stft(y_segment, n_fft=2048)
    magnitude = np.abs(stft)
    freqs = librosa.fft_frequencies(sr=sr, n_fft=2048)
    
    # 주파수 대역별 에너지
    freq_bands = {
        '저주파 (20-300Hz)': (20, 300),
        '말소리 (300-2000Hz)': (300, 2000), 
        '고주파 (2000-8000Hz)': (2000, 8000),
        '초고주파 (8000Hz+)': (8000, sr//2)
    }
    
    print(f"\\n🎵 주파수 대역별 에너지:")
    band_energies = {}
    
    for band_name, (low_freq, high_freq) in freq_bands.items():
        low_idx = np.searchsorted(freqs, low_freq)
        high_idx = np.searchsorted(freqs, high_freq)
        band_energy = np.mean(magnitude[low_idx:high_idx])
        band_energies[band_name] = band_energy
        print(f"   - {band_name}: {band_energy:.4f}")
    
    # 3. 노이즈 유형 추정
    print(f"\\n🎯 노이즈 유형 추정:")
    
    total_energy = sum(band_energies.values())
    speech_ratio = band_energies['말소리 (300-2000Hz)'] / total_energy if total_energy > 0 else 0
    low_freq_ratio = band_energies['저주파 (20-300Hz)'] / total_energy if total_energy > 0 else 0
    high_freq_ratio = band_energies['고주파 (2000-8000Hz)'] / total_energy if total_energy > 0 else 0
    
    if speech_ratio > 0.4:
        noise_type = "🗣️  말소리 포함 구간"
    elif high_freq_ratio > 0.3:
        noise_type = "📺 TV/전자기기 소음"
    elif low_freq_ratio > 0.4:
        noise_type = "🌬️  환경/배경 소음"
    else:
        noise_type = "🔇 조용한 구간"
    
    print(f"   - 추정 유형: {noise_type}")
    print(f"   - 말소리 비율: {speech_ratio:.1%}")
    print(f"   - 저주파 비율: {low_freq_ratio:.1%}")
    print(f"   - 고주파 비율: {high_freq_ratio:.1%}")
    
    # 4. 노이즈 제거 권장사항
    print(f"\\n💡 노이즈 제거 권장사항:")
    
    if rms_level > 0.1:
        print("   ⚠️  매우 높은 노이즈 레벨 - 강력한 제거 필요")
        reduction_strength = "strong"
    elif rms_level > 0.05:
        print("   ⚡ 보통 노이즈 레벨 - 중간 강도 제거")
        reduction_strength = "medium"
    else:
        print("   ✅ 낮은 노이즈 레벨 - 약한 제거로 충분")
        reduction_strength = "light"
    
    if speech_ratio > 0.3:
        print("   🎤 말소리 감지 - 2단계 처리 권장 (노이즈 제거 + 고주파 필터)")
    
    if high_freq_ratio > 0.4:
        print("   📻 고주파 소음 감지 - 2000Hz 로우패스 필터 권장")
    
    # 5. 시각화
    plot_noise_analysis(y_segment, sr, start_time_minutes, band_energies, noise_type)
    
    return {
        'rms_level': rms_level,
        'noise_type': noise_type,
        'reduction_strength': reduction_strength,
        'band_energies': band_energies,
        'speech_ratio': speech_ratio
    }

def plot_noise_analysis(y_segment, sr, start_time_minutes, band_energies, noise_type):
    """
    노이즈 분석 결과 시각화
    """
    
    fig, axes = plt.subplots(2, 2, figsize=(15, 10))
    fig.suptitle(f'노이즈 분석 결과 - {start_time_minutes:.1f}분 구간\\n{noise_type}', fontsize=16)
    
    # 1. 파형
    time_axis = np.linspace(0, len(y_segment)/sr/60, len(y_segment))
    axes[0,0].plot(time_axis, y_segment)
    axes[0,0].set_title('파형 (Waveform)')
    axes[0,0].set_xlabel('시간 (분)')
    axes[0,0].set_ylabel('진폭')
    axes[0,0].grid(True, alpha=0.3)
    
    # 2. 스펙트로그램
    stft = librosa.stft(y_segment)
    db = librosa.amplitude_to_db(np.abs(stft), ref=np.max)
    img = librosa.display.specshow(db, sr=sr, x_axis='time', y_axis='hz', ax=axes[0,1])
    axes[0,1].set_title('스펙트로그램')
    axes[0,1].set_ylim(0, 4000)  # 4kHz까지만 표시
    plt.colorbar(img, ax=axes[0,1], format='%+2.0f dB')
    
    # 3. 주파수 스펙트럼
    fft = np.fft.fft(y_segment)
    freqs = np.fft.fftfreq(len(y_segment), 1/sr)
    magnitude = np.abs(fft)
    
    # 양의 주파수만 표시
    positive_freqs = freqs[:len(freqs)//2]
    positive_magnitude = magnitude[:len(magnitude)//2]
    
    axes[1,0].plot(positive_freqs, 20*np.log10(positive_magnitude + 1e-10))
    axes[1,0].set_title('주파수 스펙트럼')
    axes[1,0].set_xlabel('주파수 (Hz)')
    axes[1,0].set_ylabel('크기 (dB)')
    axes[1,0].set_xlim(0, 4000)
    axes[1,0].grid(True, alpha=0.3)
    
    # 4. 주파수 대역별 에너지
    bands = list(band_energies.keys())
    energies = list(band_energies.values())
    
    bars = axes[1,1].bar(range(len(bands)), energies, color=['blue', 'orange', 'red', 'purple'])
    axes[1,1].set_title('주파수 대역별 에너지')
    axes[1,1].set_xticks(range(len(bands)))
    axes[1,1].set_xticklabels(bands, rotation=45, ha='right')
    axes[1,1].set_ylabel('에너지')
    
    # 막대 위에 값 표시
    for bar, energy in zip(bars, energies):
        height = bar.get_height()
        axes[1,1].text(bar.get_x() + bar.get_width()/2., height,
                       f'{energy:.3f}', ha='center', va='bottom')
    
    plt.tight_layout()
    plt.show()

# 사용 예시 함수
def quick_segment_check():
    """
    자주 사용하는 구간들을 빠르게 체크
    """
    print("🚀 주요 구간 빠른 체크")
    
    segments = [
        (0, 5, "시작 5분"),
        (10, 5, "10-15분"), 
        (20, 5, "20-25분"),
        (60, 5, "1시간 후 5분")
    ]
    
    for start, duration, description in segments:
        print(f"\\n{'='*50}")
        print(f"📍 {description} 구간 분석")
        try:
            result = segment_noise_analysis(audio_file, 
                                          start_minutes=start, 
                                          duration_minutes=duration)
            if result:
                print(f"✅ {description}: {result['noise_type']} - {result['reduction_strength']} 제거 권장")
        except Exception as e:
            print(f"❌ {description}: 분석 실패 - {e}")
    
    print(f"\\n🎯 결론: 가장 조용한 구간을 노이즈 프로파일로 사용하세요!")


In [ ]:
# 🎯 구간별 노이즈 분석 사용 예시

print("📚 사용법 가이드:")
print()
print("1️⃣ 특정 시간 구간 분석:")
print("   segment_noise_analysis(audio_file, start_minutes=10, end_minutes=30)")
print("   # 10분부터 30분까지 20분간 분석")
print()
print("2️⃣ 시작 시간 + 지속 시간:")  
print("   segment_noise_analysis(audio_file, start_minutes=15, duration_minutes=5)")
print("   # 15분부터 5분 동안 분석")
print()
print("3️⃣ 시작 시간부터 끝까지:")
print("   segment_noise_analysis(audio_file, start_minutes=60)")
print("   # 1시간부터 파일 끝까지")
print()
print("4️⃣ 주요 구간 빠른 체크:")
print("   quick_segment_check()")
print("   # 여러 구간을 한 번에 체크")

# 실제 사용 예시 (주석 해제해서 사용)
# result = segment_noise_analysis(audio_file, start_minutes=0, duration_minutes=5)

print()
print("🎯 출력 정보:")
print("   - 음량 분석 (RMS, 최대 진폭, 다이내믹 레인지)")
print("   - 주파수 대역별 에너지 분석") 
print("   - 노이즈 유형 추정 (말소리/TV/환경음)")
print("   - 노이즈 제거 강도 권장사항")
print("   - 4가지 그래프 (파형, 스펙트로그램, 주파수 스펙트럼, 대역별 에너지)")


In [ ]:
def segment_noise_analysis_v2(audio_input, sr=22050, start_minutes=0, end_minutes=None, duration_minutes=None, audio_offset_minutes=0):
    """
    오디오 파일 경로 또는 이미 로드된 오디오 데이터로 구간별 노이즈 분석
    
    Parameters:
    - audio_input: 파일 경로(str/Path) 또는 오디오 데이터(numpy array)
    - sr: 샘플링 레이트
    - start_minutes: 분석 시작 시간 (분) - audio_input 기준
    - end_minutes: 분석 종료 시간 (분) 
    - duration_minutes: 분석 지속 시간 (분)
    - audio_offset_minutes: 원본 파일에서 audio_input이 시작되는 시점 (표시용)
    
    Examples:
    # 1. 파일 경로로 사용 (기존 방식)
    segment_noise_analysis_v2('audio.wav', start_minutes=10, duration_minutes=5)
    
    # 2. 이미 처리된 오디오 데이터로 사용 (새로운 방식)
    sleep_audio, sr = extract_sleep_portion(data_path, start_minutes=20)
    segment_noise_analysis_v2(sleep_audio, sr=sr, start_minutes=0, duration_minutes=5, audio_offset_minutes=20)
    """
    
    print(f"🔍 구간별 노이즈 분석 시작 (v2)")
    
    # 입력 타입 확인 및 처리
    if isinstance(audio_input, (str, Path)):
        # 파일 경로인 경우
        print(f"파일에서 로드: {audio_input}")
        y_full, sr = librosa.load(audio_input, sr=sr)
        audio_source = "파일"
    else:
        # 이미 로드된 오디오 데이터인 경우
        print(f"사전 처리된 오디오 데이터 사용")
        y_full = np.array(audio_input)
        audio_source = "처리된 데이터"
    
    total_duration_minutes = len(y_full) / sr / 60
    print(f"{audio_source} 길이: {total_duration_minutes:.1f}분 ({len(y_full)/sr/3600:.1f}시간)")
    
    if audio_offset_minutes > 0:
        print(f"원본 파일 기준 오프셋: +{audio_offset_minutes:.1f}분")
    
    # 구간 계산 (y_full 기준)
    start_sample = int(start_minutes * 60 * sr)
    
    if end_minutes is not None:
        end_sample = int(end_minutes * 60 * sr)
        actual_duration = end_minutes - start_minutes
    elif duration_minutes is not None:
        end_sample = start_sample + int(duration_minutes * 60 * sr)
        actual_duration = duration_minutes
    else:
        # 끝까지 분석
        end_sample = len(y_full)
        actual_duration = (len(y_full) - start_sample) / sr / 60
    
    # 범위 체크
    end_sample = min(end_sample, len(y_full))
    
    # 원본 파일 기준 시간 계산 (표시용)
    original_start_time = start_minutes + audio_offset_minutes
    original_end_time = original_start_time + actual_duration
    
    print(f"분석 구간:")
    print(f"  - 현재 데이터 기준: {start_minutes:.1f}분 ~ {start_minutes + actual_duration:.1f}분")
    if audio_offset_minutes > 0:
        print(f"  - 원본 파일 기준: {original_start_time:.1f}분 ~ {original_end_time:.1f}분")
    print(f"  - 분석 길이: {actual_duration:.1f}분")
    
    # 해당 구간만 추출
    y_segment = y_full[start_sample:end_sample]
    segment_duration = len(y_segment) / sr
    
    if len(y_segment) == 0:
        print("❌ 오류: 지정된 구간이 비어있거나 범위를 벗어났습니다.")
        return None
    
    print(f"추출된 구간 길이: {segment_duration/60:.1f}분")
    
    # 분석 실행 (원본 시간 정보 전달)
    return analyze_noise_characteristics(y_segment, sr, original_start_time)


In [ ]:
# 🎯 사용자 상황에 맞는 사용 예시

print("🚀 이미 처리된 sleep_audio로 노이즈 분석하기")
print()

# 1. 기존 코드 (이미 완료됨)
# sleep_audio, sr = extract_sleep_portion(data_path, start_minutes=20)

print("1️⃣ 수면 구간의 처음 5분 분석:")
print("   segment_noise_analysis_v2(sleep_audio, sr=sr, start_minutes=0, duration_minutes=5, audio_offset_minutes=20)")
print("   # sleep_audio의 처음 5분 (원본 파일 기준 20-25분)")
print()

print("2️⃣ 수면 구간의 1시간 후 10분 분석:")  
print("   segment_noise_analysis_v2(sleep_audio, sr=sr, start_minutes=60, duration_minutes=10, audio_offset_minutes=20)")
print("   # sleep_audio의 60분 후 (원본 파일 기준 80-90분)")
print()

print("3️⃣ 수면 구간 전체 개요:")
print("   segment_noise_analysis_v2(sleep_audio, sr=sr, start_minutes=0, duration_minutes=60, audio_offset_minutes=20)")
print("   # 수면 구간의 첫 1시간 분석")
print()

print("💡 audio_offset_minutes=20의 의미:")
print("   - sleep_audio는 원본 파일의 20분 후부터 시작")
print("   - 시각화에서 '원본 파일 기준' 시간을 정확히 표시")
print("   - 예: sleep_audio의 0분 = 원본의 20분")

print()
print("🎬 실제 사용 (주석 해제):")
print("# result = segment_noise_analysis_v2(sleep_audio, sr=sr, start_minutes=0, duration_minutes=5, audio_offset_minutes=20)")

# 간단한 테스트 (sleep_audio가 있다고 가정)
print()
print("📊 분석 결과에서 얻을 수 있는 정보:")
print("   ✅ 어느 구간이 가장 조용한지")
print("   ✅ 노이즈 제거 강도는 어느 정도가 적당한지") 
print("   ✅ 말소리가 남아있는 구간은 어디인지")
print("   ✅ 코골이 분석에 적합한 구간은 어디인지")


In [ ]:
def quick_sleep_analysis(sleep_audio, sr, audio_offset_minutes=20):
    """
    처리된 sleep_audio의 주요 구간들을 빠르게 분석
    """
    print("🌙 수면 구간 주요 포인트 빠른 분석")
    print(f"수면 데이터 길이: {len(sleep_audio)/sr/60:.1f}분")
    print()
    
    # 분석할 구간들 정의 (sleep_audio 기준)
    segments = [
        (0, 10, "수면 초기 10분"),
        (30, 15, "수면 30분 후"), 
        (60, 15, "수면 1시간 후"),
        (120, 15, "수면 2시간 후"),
        (240, 15, "수면 4시간 후")
    ]
    
    results = {}
    
    for start, duration, description in segments:
        # 범위 체크
        max_duration = (len(sleep_audio)/sr/60) - start
        if max_duration <= 0:
            print(f"⏭️  {description}: 범위 초과 - 건너뜀")
            continue
            
        actual_duration = min(duration, max_duration)
        
        print(f"\\n{'='*50}")
        print(f"📍 {description} 분석 중...")
        
        try:
            result = segment_noise_analysis_v2(
                sleep_audio, 
                sr=sr,
                start_minutes=start, 
                duration_minutes=actual_duration,
                audio_offset_minutes=audio_offset_minutes
            )
            
            if result:
                results[description] = result
                print(f"✅ {description}: {result['noise_type']} - {result['reduction_strength']} 강도 권장")
                
        except Exception as e:
            print(f"❌ {description}: 분석 실패 - {e}")
    
    # 결과 요약
    print(f"\\n🎯 === 분석 요약 ===")
    
    if results:
        # 가장 조용한 구간 찾기
        quietest = min(results.items(), key=lambda x: x[1]['rms_level'])
        noisiest = max(results.items(), key=lambda x: x[1]['rms_level'])
        
        print(f"🔇 가장 조용한 구간: {quietest[0]} (RMS: {quietest[1]['rms_level']:.4f})")
        print(f"🔊 가장 시끄러운 구간: {noisiest[0]} (RMS: {noisiest[1]['rms_level']:.4f})")
        
        # 말소리 구간 찾기
        speech_segments = [name for name, result in results.items() 
                          if result['speech_ratio'] > 0.3]
        
        if speech_segments:
            print(f"말소리 감지 구간: {', '.join(speech_segments)}")
        else:
            print(f"말소리 없는 깨끗한 수면 구간")
            
        # 권장사항
        print(f"\\n 권장사항:")
        print(f"   - 노이즈 프로파일: {quietest[0]} 구간 사용")
        
        strong_noise_segments = [name for name, result in results.items() 
                               if result['reduction_strength'] == 'strong']
        if strong_noise_segments:
            print(f"   - 강력한 노이즈 제거 필요: {', '.join(strong_noise_segments)}")
            
    else:
        print("분석된 구간이 없습니다.")
    
    return results

# 실제 사용 예시 (주석 해제해서 사용)
print("\\n 사용법:")
print("results = quick_sleep_analysis(sleep_audio, sr, audio_offset_minutes=20)")
print("# 수면 구간의 주요 포인트들을 한 번에 분석")


In [ ]:
import librosa
import numpy as np
import noisereduce as nr
from scipy import signal
import matplotlib.pyplot as plt
from scipy.signal import butter, filtfilt, savgol_filter

# 심각한 노이즈 상황을 위한 강력한 노이즈 제거

def aggressive_noise_reduction(audio_file, sr=22050, noise_duration_minutes=20):
    """
    심각한 배경 노이즈를 위한 강력한 다단계 노이즈 제거
    """
    print("=== 강력한 노이즈 제거 시작 ===")
    
    # 오디오 로드
    y, sr = librosa.load(audio_file, sr=sr)
    print(f"원본 오디오 길이: {len(y)/sr/60:.1f}분")
    
    # 노이즈 샘플 추출 (처음 20분)
    noise_duration_seconds = noise_duration_minutes * 60
    noise_end_sample = int(noise_duration_seconds * sr)
    noise_end_sample = min(noise_end_sample, len(y))
    noise_sample = y[:noise_end_sample]
    
    # 1단계: 강력한 스펙트럼 차감법
    print("1단계: 강력한 스펙트럼 차감 (80% 제거)...")
    stage1 = nr.reduce_noise(
        y=y,
        sr=sr,
        y_noise=noise_sample,
        stationary=False,
        prop_decrease=0.8,  # 80% 강력 제거
        n_grad_freq=4,      # 주파수 그라디언트 증가
        n_grad_time=8,      # 시간 그라디언트 증가
        n_fft=2048,         # FFT 크기 증가
        win_length=2048,
        hop_length=512
    )
    
    # 2단계: 적응적 노이즈 제거
    print("2단계: 적응적 노이즈 제거...")
    # 더 짧은 구간의 노이즈 프로파일로 재시도
    short_noise = noise_sample[:int(5*60*sr)]  # 5분만 사용
    stage2 = nr.reduce_noise(
        y=stage1,
        sr=sr,
        y_noise=short_noise,
        stationary=True,    # 정상 노이즈로 가정
        prop_decrease=0.6,
        n_grad_freq=2,
        n_grad_time=4
    )
    
    # 3단계: 주파수 도메인 필터링
    print("3단계: 주파수 도메인 필터링...")
    # 고주파 노이즈 제거 (4000Hz 이상)
    nyquist = sr / 2
    cutoff_high = 4000 / nyquist
    b_high, a_high = butter(N=6, Wn=cutoff_high, btype='low')
    stage3 = filtfilt(b_high, a_high, stage2)
    
    # 극저주파 노이즈 제거 (20Hz 이하)
    cutoff_low = 20 / nyquist  
    b_low, a_low = butter(N=4, Wn=cutoff_low, btype='high')
    stage3 = filtfilt(b_low, a_low, stage3)
    
    # 4단계: 스무딩 (급격한 변화 제거)
    print("4단계: 신호 스무딩...")
    # Savitzky-Golay 필터로 스무딩
    window_length = min(51, len(stage3)//10)  # 창 크기 조정
    if window_length % 2 == 0:  # 홀수로 만들기
        window_length += 1
    if window_length >= 3:
        stage4 = savgol_filter(stage3, window_length, 3)
    else:
        stage4 = stage3
    
    print("=== 노이즈 제거 완료 ===")
    return stage4, sr, {
        'stage1': stage1, 
        'stage2': stage2, 
        'stage3': stage3, 
        'stage4': stage4
    }

def visualize_noise_reduction_process(original, stages, sr):
    """
    노이즈 제거 과정을 시각화
    """
    fig, axes = plt.subplots(5, 2, figsize=(18, 20))
    
    # 시간 영역 비교
    time = np.linspace(0, len(original)/sr, len(original))
    
    # 원본
    axes[0,0].plot(time, original)
    axes[0,0].set_title('원본 오디오 (시간)')
    axes[0,0].set_ylabel('Amplitude')
    
    # 각 단계
    for i, (stage_name, audio) in enumerate(stages.items(), 1):
        if len(audio) != len(original):
            time_stage = np.linspace(0, len(audio)/sr, len(audio))
        else:
            time_stage = time
        axes[i,0].plot(time_stage, audio)
        axes[i,0].set_title(f'{stage_name} (시간)')
        axes[i,0].set_ylabel('Amplitude')
    
    # 주파수 영역 비교
    # 원본 스펙트럼
    freqs_orig, psd_orig = plt.mlab.psd(original, Fs=sr, NFFT=2048)
    axes[0,1].semilogy(freqs_orig, psd_orig)
    axes[0,1].set_title('원본 오디오 (주파수)')
    axes[0,1].set_ylabel('Power Spectral Density')
    axes[0,1].set_xlim(0, 5000)
    
    # 각 단계 스펙트럼
    for i, (stage_name, audio) in enumerate(stages.items(), 1):
        freqs, psd = plt.mlab.psd(audio, Fs=sr, NFFT=2048)
        axes[i,1].semilogy(freqs, psd)
        axes[i,1].set_title(f'{stage_name} (주파수)')
        axes[i,1].set_ylabel('Power Spectral Density')
        axes[i,1].set_xlim(0, 5000)
    
    axes[-1,0].set_xlabel('Time (s)')
    axes[-1,1].set_xlabel('Frequency (Hz)')
    
    plt.tight_layout()
    plt.show()

def analyze_noise_reduction_effectiveness(original, cleaned, sr):
    """
    노이즈 제거 효과 분석
    """
    print("=== 노이즈 제거 효과 분석 ===")
    
    # SNR 개선 계산
    original_power = np.mean(original**2)
    cleaned_power = np.mean(cleaned**2)
    
    # RMS 비교
    original_rms = np.sqrt(np.mean(original**2))
    cleaned_rms = np.sqrt(np.mean(cleaned**2))
    
    print(f"원본 RMS: {original_rms:.4f}")
    print(f"처리 후 RMS: {cleaned_rms:.4f}")
    print(f"RMS 감소율: {(1-cleaned_rms/original_rms)*100:.1f}%")
    
    # 주파수별 에너지 비교
    freqs_orig, psd_orig = plt.mlab.psd(original, Fs=sr)
    freqs_clean, psd_clean = plt.mlab.psd(cleaned, Fs=sr)
    
    # 노이즈 영역 (고주파) 에너지 비교
    high_freq_mask = freqs_orig > 2000
    orig_high_energy = np.sum(psd_orig[high_freq_mask])
    clean_high_energy = np.sum(psd_clean[high_freq_mask])
    
    print(f"고주파 노이즈 제거율: {(1-clean_high_energy/orig_high_energy)*100:.1f}%")
    
    # 신호 영역 (저주파) 보존율
    low_freq_mask = freqs_orig < 1000
    orig_low_energy = np.sum(psd_orig[low_freq_mask]) 
    clean_low_energy = np.sum(psd_clean[low_freq_mask])
    
    print(f"저주파 신호 보존율: {clean_low_energy/orig_low_energy*100:.1f}%")

# 사용 예시
def process_noisy_audio(audio_file):
    """
    노이즈가 심한 오디오 처리 전체 과정
    """
    # 원본 로드
    original, sr = librosa.load(audio_file, sr=22050)
    
    # 강력한 노이즈 제거
    cleaned, sr, stages = aggressive_noise_reduction(audio_file)
    
    # 과정 시각화
    visualize_noise_reduction_process(original, stages, sr)
    
    # 효과 분석
    analyze_noise_reduction_effectiveness(original, cleaned, sr)
    
    return original, cleaned, sr

print("강력한 노이즈 제거 시스템이 준비되었습니다!")
print("사용법: original, cleaned, sr = process_noisy_audio('your_audio_file.wav')")


In [ ]:
# 극심한 노이즈를 위한 추가 도구들

def emergency_noise_reduction(audio_file, sr=22050):
    """
    극심한 노이즈 상황을 위한 응급 처치
    """
    print("🚨 응급 노이즈 제거 모드 🚨")
    
    y, sr = librosa.load(audio_file, sr=sr)
    
    # 초강력 노이즈 제거 (90% 제거)
    noise_sample = y[:int(20*60*sr)]  # 20분 노이즈
    
    ultra_clean = nr.reduce_noise(
        y=y,
        sr=sr,
        y_noise=noise_sample,
        stationary=False,
        prop_decrease=0.9,  # 90% 제거!
        n_grad_freq=6,      # 최대 그라디언트
        n_grad_time=12,     # 최대 시간 스무딩
        n_fft=4096,         # 큰 FFT
        win_length=4096,
        hop_length=256
    )
    
    print("✅ 90% 강력 제거 완료")
    return ultra_clean, sr

def quick_noise_check(audio_file):
    """
    빠른 노이즈 레벨 체크
    """
    y, sr = librosa.load(audio_file, sr=22050, duration=60)  # 1분만 로드
    
    # 기본 통계
    rms = np.sqrt(np.mean(y**2))
    max_amp = np.max(np.abs(y))
    
    print("=== 오디오 품질 체크 ===")
    print(f"RMS 레벨: {rms:.4f}")
    print(f"최대 진폭: {max_amp:.4f}")
    print(f"다이나믹 레인지: {20*np.log10(max_amp/rms):.1f} dB")
    
    # 노이즈 심각도 판단
    if rms > 0.1:
        print("🔴 노이즈 매우 심함 - emergency_noise_reduction 사용")
    elif rms > 0.05:
        print("🟡 노이즈 심함 - aggressive_noise_reduction 사용")
    else:
        print("🟢 노이즈 보통 - 일반 처리 가능")
    
    return rms, max_amp

def save_cleaned_audio(cleaned_audio, sr, output_filename):
    """
    처리된 오디오 저장
    """
    import soundfile as sf
    
    # 정규화 (선택사항)
    normalized = cleaned_audio / np.max(np.abs(cleaned_audio)) * 0.8
    
    sf.write(output_filename, normalized, sr)
    print(f"✅ 정제된 오디오 저장: {output_filename}")

def compare_before_after(original_file, cleaned_audio, sr):
    """
    처리 전후 간단 비교
    """
    original, _ = librosa.load(original_file, sr=sr, duration=30)  # 30초만
    cleaned_sample = cleaned_audio[:len(original)]
    
    fig, axes = plt.subplots(2, 2, figsize=(15, 10))
    
    # 시간 영역
    time = np.linspace(0, len(original)/sr, len(original))
    axes[0,0].plot(time, original)
    axes[0,0].set_title('원본 (30초)')
    axes[0,0].set_ylabel('Amplitude')
    
    axes[1,0].plot(time, cleaned_sample)
    axes[1,0].set_title('처리 후 (30초)')
    axes[1,0].set_ylabel('Amplitude')
    axes[1,0].set_xlabel('Time (s)')
    
    # 주파수 영역
    freqs_orig, psd_orig = plt.mlab.psd(original, Fs=sr)
    freqs_clean, psd_clean = plt.mlab.psd(cleaned_sample, Fs=sr)
    
    axes[0,1].semilogy(freqs_orig, psd_orig)
    axes[0,1].set_title('원본 스펙트럼')
    axes[0,1].set_ylabel('Power')
    axes[0,1].set_xlim(0, 5000)
    
    axes[1,1].semilogy(freqs_clean, psd_clean)
    axes[1,1].set_title('처리 후 스펙트럼')
    axes[1,1].set_ylabel('Power')
    axes[1,1].set_xlabel('Frequency (Hz)')
    axes[1,1].set_xlim(0, 5000)
    
    plt.tight_layout()
    plt.show()

# 편의 함수들
def step_by_step_cleaning(audio_file):
    """
    단계별로 확인하면서 노이즈 제거
    """
    print("=== 단계별 노이즈 제거 ===")
    
    # 1. 먼저 노이즈 레벨 체크
    rms, max_amp = quick_noise_check(audio_file)
    
    # 2. 노이즈 레벨에 따라 방법 선택
    if rms > 0.1:
        print("응급 처리 모드 선택")
        cleaned, sr = emergency_noise_reduction(audio_file)
    else:
        print("일반 강력 처리 모드 선택")
        cleaned, sr, _ = aggressive_noise_reduction(audio_file)
    
    # 3. 결과 비교
    compare_before_after(audio_file, cleaned, sr)
    
    # 4. 저장 여부 확인
    save_option = input("처리된 오디오를 저장하시겠습니까? (y/n): ")
    if save_option.lower() == 'y':
        output_name = input("출력 파일명 (예: cleaned_audio.wav): ")
        save_cleaned_audio(cleaned, sr, output_name)
    
    return cleaned, sr

print("🔧 추가 도구들이 준비되었습니다!")
print("1. quick_noise_check('file.wav') - 빠른 노이즈 레벨 체크")
print("2. emergency_noise_reduction('file.wav') - 응급 90% 제거")
print("3. step_by_step_cleaning('file.wav') - 단계별 처리")
print("4. save_cleaned_audio(audio, sr, 'output.wav') - 저장")
